# LangGraph Critic Loop — Summarizer ↔ Reviewer
## A Hands-On Workshop

The **critic loop** is one of the most practical multi-agent patterns: one LLM agent generates an output, a second LLM agent critiques it, and the generator revises until quality is met or a round cap is reached. It is the basis for self-improving summarizers, code reviewers, and document drafters.

By the end of this session you will understand *why* accumulated message state is essential for a feedback loop, *how* LangGraph's `StateGraph` wires the cycle without infinite recursion, and *how* to steer an in-flight graph using `MemorySaver` and `thread_id`.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Critic loop pattern and why it exists |
| 2 | **State Design** — `Annotated[list, operator.add]` vs replace semantics |
| 3 | **Graph Construction** — Nodes, conditional edges, entry point |
| 4 | **Running the Loop** — Single-shot invocation and trace inspection |
| 5 | **Multi-Turn Refinement** — `MemorySaver` + `thread_id` for user steering |
| 6 | **Stopping Conditions** — Iteration caps and quality gates |
| ★ | **Advanced Patterns** — Round counter, strict reviewer, streaming |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the repo's `requirements.txt`
- An `OPENAI_API_KEY` in `.env` (or Colab Secrets)
- No external files required — the notebook uses inline text for the document

### Companion Examples
- **Example 4** `4-basic-react-agent` — single-agent ReAct with tool calling
- **Example 6** `6-multi-agent-graph` — routing between specialist agents
- **Example 18** `18-self-reflecting-agent` — reflexion: an agent critiques its own output

### Key References
> Yao, S., Zhao, J., Yu, D., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
>
> Shinn, N., Cassano, F., et al. (2023). *Reflexion: Language Agents with Verbal Reinforcement Learning.* NeurIPS 2023. https://arxiv.org/abs/2303.11366
>
> LangGraph StateGraph concepts: https://langchain-ai.github.io/langgraph/concepts/
>
> LangGraph `MemorySaver` persistence: https://langchain-ai.github.io/langgraph/concepts/persistence/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or malformed. "
    "Local: add it to .env. Colab: add it to Secrets."
)
print(f"API key OK — {key[:8]}...{key[-4:]}")

---

## Part 1 — The Critic Loop Pattern

---

### The Problem

A single LLM call produces output of unpredictable quality. The model has no way to check its own work against an external standard unless you explicitly ask it to critique, and then it often just agrees with itself.

**The Critic Loop** solves this by separating roles:
- A **Generator** (Summarizer) produces output
- A **Critic** (Reviewer) evaluates it against the source and proposes improvements
- The Generator revises using the critique
- This cycle repeats until a stopping condition is met

---

### Raw Prompt Loop vs LangGraph Critic Loop

| Dimension | Raw prompt loop | LangGraph Critic Loop |
|-----------|----------------|-----------------------|
| **State management** | Manual — you concatenate messages in Python | Automatic — `StateGraph` accumulates via `operator.add` |
| **History visibility** | Easy to lose context across iterations | Every node sees the full message history |
| **Stopping condition** | `if` check in your loop | `should_continue` conditional edge — graph-native |
| **Multi-turn user steering** | Re-run the whole loop | `MemorySaver` + `thread_id` — resume mid-conversation |
| **Observability** | `print()` statements | LangGraph trace / LangSmith integration |
| **Async / streaming** | Manual `asyncio` | Built-in `astream_events` |

---

### Graph Architecture

```
  ┌─────────────────────────────────────────────────────────┐
  │                    SummaryAgentState                     │
  │   messages: Annotated[list[AnyMessage], operator.add]   │
  └─────────────────────────────────────────────────────────┘
                            │
                       START / invoke()
                            │
                            ▼
                  ┌──────────────────┐
                  │   summarizer     │  SystemMessage(summarizer_prompt)
                  │  (generate_     │  + full message history
                  │   summary)      │  → llm.invoke()
                  └────────┬─────────┘  appends AIMessage to state
                           │
                  should_continue?
                    ╱           ╲
               True              False
                ╱                   ╲
               ▼                    ▼
     ┌──────────────────┐          END
     │    reviewer      │
     │  (review_       │  SystemMessage(reviewer_prompt)
     │   summary)      │  + full message history (incl. draft)
     └────────┬─────────┘  → llm.invoke()
              │              appends critique AIMessage to state
              │
              └──────────────────────► summarizer  (loop)


  Stopping condition (default): len(state["messages"]) > 4
  = 1 HumanMessage + 2 messages/round × 2 rounds = 5 total → stop
```

### Key Concepts

| Concept | Definition |
|---------|------------|
| **StateGraph** | LangGraph's graph class — nodes are Python functions, edges define execution flow |
| **TypedDict state** | A Python `TypedDict` describing the data shared across all nodes |
| **`operator.add`** | List merge reducer — new messages are *appended*, not replaced |
| **`add_conditional_edges`** | Routes to different next nodes based on a condition function |
| **`MemorySaver`** | In-memory checkpointer — persists state across `invoke()` calls on the same `thread_id` |
| **`thread_id`** | Unique identifier for a conversation — determines which checkpoint to resume |

---

## Part 2 — State Design: `operator.add` vs Replace

---

### Why Accumulating State Is Essential

In LangGraph, when a node returns `{"messages": [...]}`, the default behaviour is to **replace** the existing `messages` field with the new value. This would be catastrophic for a critic loop:

```
Round 1:
  summarizer returns {"messages": [AIMessage("Draft 1")]}
  state.messages = [HumanMessage(doc), AIMessage("Draft 1")]  OK

  reviewer returns {"messages": [AIMessage("Needs numbers")]}
  state.messages = [AIMessage("Needs numbers")]               WRONG!
  # The source document and Draft 1 are GONE
  # Summarizer on round 2 has NO CONTEXT
```

With `Annotated[list[AnyMessage], operator.add]`, each return is **appended**:

```
Round 1:
  Initial:   [HumanMessage(doc)]
  After gen: [HumanMessage(doc), AIMessage("Draft 1")]
  After rev: [HumanMessage(doc), AIMessage("Draft 1"), AIMessage("Needs numbers")]

Round 2:
  After gen: [..., AIMessage("Needs numbers"), AIMessage("Draft 2")]
  After rev: [..., AIMessage("Better, but fix X")]
```

The summarizer on round 2 can read both the source document **and** the reviewer's critique.

In [ ]:
# ─── 2-A: operator.add — how message accumulation works ───────────────────────
import operator

# Simulate two rounds of the critic loop
state_messages = ["HumanMessage: source document text"]

# Round 1 — summarizer returns its draft
from_summarizer_r1 = ["AIMessage: Draft 1 — short summary"]
state_messages = operator.add(state_messages, from_summarizer_r1)
print("After summarizer round 1:", state_messages)

# Round 1 — reviewer returns critique
from_reviewer_r1 = ["AIMessage: Needs more numbers, missing battery range"]
state_messages = operator.add(state_messages, from_reviewer_r1)
print("After reviewer round 1: ", state_messages)

# Round 2 — summarizer sees EVERYTHING and can revise
from_summarizer_r2 = ["AIMessage: Draft 2 — improved summary with 60-80km range"]
state_messages = operator.add(state_messages, from_summarizer_r2)
print("After summarizer round 2:", state_messages)

print(f"\nTotal messages in state: {len(state_messages)}")
print("Summarizer on round 2 has full context: source + draft 1 + critique")

In [ ]:
# ─── 2-B: TypedDict state definition ──────────────────────────────────────────
# This is the exact state used in the production SummaryAgent from main.py.
import typing
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage


class SummaryAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


# Demonstrate the annotation — it's just metadata on the type
hints = typing.get_type_hints(SummaryAgentState, include_extras=True)
field_annotation = hints["messages"]
print("Field type:", field_annotation)
print("Reducer:   ", typing.get_args(field_annotation)[1])
print()
print("LangGraph reads this annotation to know HOW to merge node outputs.")
print("operator.add on lists → append-only, never replace.")

---

## Part 3 — Building the Critic Loop Graph

---

### Node Design

Each node is a Python function that receives the full state and returns a **partial state update**:

```python
def generate_summary(state: SummaryAgentState) -> dict:
    messages = [SystemMessage(SUMMARIZER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    return {"messages": [result]}   # operator.add appends this to state
```

Both `generate_summary` and `review_summary` follow the same pattern — prepend a system prompt then invoke the LLM. The only difference is which system prompt is prepended.

### Conditional Edge

```python
graph.add_conditional_edges(
    "summarizer",           # source node
    should_continue,        # condition function: state → bool
    {True: "reviewer",     # True  → keep looping
     False: END}            # False → terminate
)
```

The edge only exists on `summarizer` — the reviewer **always** loops back to the summarizer. This means the graph always ends on a summarizer turn (the last message is a revised draft, not a critique).

In [ ]:
# ─── 3-A: Prompts and LLM setup ───────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage

SUMMARIZER_PROMPT = (
    "You are a document summarizer. "
    "Create a summary in under 50 words that covers the key facts. "
    "If the user has provided critique, respond with a revised version of your previous summary."
)

REVIEWER_PROMPT = (
    "You are a quality reviewer. "
    "Compare the source document and the generated summary. "
    "Check if the summary accurately reflects the contents of the document. "
    "Provide specific recommendations for improvement in under 50 words."
)

# gpt-4o-mini is the right balance of quality and cost for iterative loops
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# Source document — inline so the notebook is fully self-contained
SOURCE_TEXT = (
    "EcoSprint X3 Specification\n"
    "The EcoSprint X3 is a lightweight electric bicycle for urban commuters. "
    "Motor: 250W brushless mid-drive. Battery: 36V 10Ah lithium-ion, range 60-80km. "
    "Frame: 6061 aluminum, weight 18kg. Brakes: hydraulic disc front and rear. "
    "Display: backlit LCD. Charging: 3.5 hours. Max speed: 25 km/h. "
    "Foldable. IP54 water resistance. Price: EUR 1,299."
)

print("LLM ready:", llm.model_name)
print(f"Source document: {len(SOURCE_TEXT)} chars")

In [ ]:
# ─── 3-B: Node functions ──────────────────────────────────────────────────────
# Both nodes follow the same pattern:
#   1. Prepend a role-specific SystemMessage
#   2. Invoke the LLM with the full message history
#   3. Return {"messages": [result]} — operator.add appends to state


def generate_summary(state: SummaryAgentState) -> dict:
    """Summarizer node: drafts or revises a summary using the full message history."""
    messages = [SystemMessage(SUMMARIZER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    print(f"  [summarizer] {result.content[:120]}")
    return {"messages": [result]}


def review_summary(state: SummaryAgentState) -> dict:
    """Reviewer node: critiques the latest summary and suggests improvements."""
    messages = [SystemMessage(REVIEWER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    print(f"  [reviewer]   {result.content[:120]}")
    return {"messages": [result]}


def should_continue(state: SummaryAgentState) -> bool:
    """Stopping condition: stop after 2 full rounds (1 input + 4 generated messages)."""
    # Each round adds 2 messages (one from summarizer, one from reviewer).
    # Initial state has 1 message (the HumanMessage with the source doc).
    # After 2 rounds: 1 + 4 = 5 messages total → should_continue returns False.
    return len(state["messages"]) <= 4


print("Nodes defined: generate_summary, review_summary")
print("Condition: should_continue (stops at 5 messages = 2 full rounds)")

In [ ]:
# ─── 3-C: Assemble the StateGraph ─────────────────────────────────────────────
from langgraph.graph import END, StateGraph
from langgraph.checkpoint.memory import MemorySaver

graph_builder = StateGraph(SummaryAgentState)

# Add nodes
graph_builder.add_node("summarizer", generate_summary)
graph_builder.add_node("reviewer",   review_summary)

# Wire edges
# summarizer → reviewer (if should_continue) or END (if stopping condition met)
graph_builder.add_conditional_edges(
    "summarizer",
    should_continue,
    {True: "reviewer", False: END},
)
# reviewer always feeds back into summarizer
graph_builder.add_edge("reviewer", "summarizer")

# Entry point
graph_builder.set_entry_point("summarizer")

# Compile with MemorySaver — enables multi-turn steering via thread_id
memory = MemorySaver()
app = graph_builder.compile(checkpointer=memory)

print("Graph compiled successfully.")
print("Edges:")
print("  START         -> summarizer")
print("  summarizer    -> reviewer (if should_continue=True)")
print("  summarizer    -> END      (if should_continue=False)")
print("  reviewer      -> summarizer (always)")

---

## Part 4 — Running the Loop and Inspecting the Trace

---

### What happens during `app.invoke()`?

```
invoke({"messages": [HumanMessage(doc)]}, config)

Execution trace:
  state = {messages: [HumanMessage(doc)]}
  ↓ summarizer
  state = {messages: [HumanMessage(doc), AIMessage(draft_1)]}
  ↓ should_continue → True (len=2)
  ↓ reviewer
  state = {messages: [..., AIMessage(critique_1)]}
  ↓ summarizer
  state = {messages: [..., AIMessage(draft_2)]}
  ↓ should_continue → True (len=4)
  ↓ reviewer
  state = {messages: [..., AIMessage(critique_2)]}
  ↓ summarizer
  state = {messages: [..., AIMessage(draft_3)]}
  ↓ should_continue → False (len=6 > 4)   STOP
  → END

Final state returned with all 6 messages.
Last AIMessage = Draft 3 (the final polished summary).
```

In [ ]:
# ─── 4-A: Single-shot invocation ──────────────────────────────────────────────
import uuid
from langchain_core.messages import AIMessage

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Starting critic loop...\n")
result = app.invoke(
    {"messages": [HumanMessage(SOURCE_TEXT)]},
    config,
)

print(f"\n=== Loop complete: {len(result['messages'])} messages in state ===")
# The last AIMessage is always the final summarizer output
for msg in reversed(result["messages"]):
    if isinstance(msg, AIMessage):
        print("\nFinal summary:")
        print(msg.content)
        break

In [ ]:
# ─── 4-B: Full message trace — see every round ────────────────────────────────
# Understanding what each agent saw and said at each round is the key
# debugging tool for critic loops.

print("=== FULL MESSAGE TRACE ===\n")
ai_indices = [i for i, m in enumerate(result["messages"]) if isinstance(m, AIMessage)]

for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__

    if msg_type == "HumanMessage":
        role = "USER      "
    elif msg_type == "AIMessage":
        ai_position = ai_indices.index(i)
        node = "summarizer" if ai_position % 2 == 0 else "reviewer"
        role = f"AI [{node}]"
    else:
        role = msg_type

    border = "─" * 60
    print(f"[{i}] {role}")
    print(border)
    print(msg.content[:300])
    if len(msg.content) > 300:
        print(f"... [{len(msg.content) - 300} more chars]")
    print()

In [ ]:
# ─── 4-C: Measure improvement across rounds ───────────────────────────────────
# A simple heuristic: count how many spec values appear in each draft.

spec_values = ["250W", "36V", "60", "80km", "18kg", "3.5", "25 km/h", "1,299"]

ai_messages = [m for m in result["messages"] if isinstance(m, AIMessage)]
# Even-indexed AI messages = summarizer drafts; odd = reviewer critiques
summarizer_drafts = [m for i, m in enumerate(ai_messages) if i % 2 == 0]

print("Spec value coverage across drafts:\n")
print(f"{'Draft':>8}  {'Coverage':>8}  {'Values found'}")
print("─" * 60)
for i, draft in enumerate(summarizer_drafts, 1):
    found = [v for v in spec_values if v in draft.content]
    pct = len(found) / len(spec_values) * 100
    bar = "\u2588" * len(found) + "\u2591" * (len(spec_values) - len(found))
    print(f"  Draft {i}:  {pct:5.0f}%   {bar}  {found}")

### Exercise 1 — Change the Source Document

Replace `MY_SOURCE_TEXT` below with any text of your choice (a product spec, an article paragraph, a policy document) and re-run the loop.

**Questions to answer:**
1. Does the reviewer catch genuine inaccuracies in the first draft?
2. How does the final draft compare to draft 1 in completeness?
3. What happens if the source text is very short (2-3 sentences)?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# TODO: replace with your own source document
MY_SOURCE_TEXT = """
Paste your own document here. Try a product spec, a policy excerpt,
or a Wikipedia paragraph. The longer and more fact-dense, the better
the critic loop can demonstrate its value.
""".strip()

# TODO: run the loop on your text
# my_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
# my_result = app.invoke({"messages": [HumanMessage(MY_SOURCE_TEXT)]}, my_config)
# print("Final summary:", my_result["messages"][-1].content)

print("Replace MY_SOURCE_TEXT above and uncomment the run block.")

---

## Part 5 — Multi-Turn Refinement with MemorySaver

---

### What MemorySaver Enables

`MemorySaver` is LangGraph's built-in in-memory checkpointer. It saves the full graph state after every node execution, keyed by `thread_id`.

When you call `app.invoke()` a second time with the **same `thread_id`**, LangGraph:
1. Loads the saved state from the checkpoint
2. Appends your new `HumanMessage` to the existing message history
3. Continues execution from the entry point with full prior context

This enables **user steering**: the human can direct the summarizer mid-session without reprocessing the source document from scratch.

```
Turn 1: invoke({HumanMessage(doc)}, thread="t1")
  runs 2 critic rounds -> saves checkpoint

Turn 2: invoke({HumanMessage("Focus on battery")}, thread="t1")
  loads checkpoint -> appends new message -> runs 2 more critic rounds
  summarizer sees: [doc, draft1, critique1, draft2, critique2, "Focus on battery"]
```

In [ ]:
# ─── 5-A: Initial run — establish baseline summary ────────────────────────────
thread_id = str(uuid.uuid4())
config_mt = {"configurable": {"thread_id": thread_id}}

print(f"Thread: {thread_id}\n")
print("=== Turn 1: Initial summary ===")
r1 = app.invoke({"messages": [HumanMessage(SOURCE_TEXT)]}, config_mt)

# Find the last AI message
final_ai = next(m for m in reversed(r1["messages"]) if isinstance(m, AIMessage))
print("\nBaseline summary:")
print(final_ai.content)
print(f"\nState has {len(r1['messages'])} messages after turn 1.")

In [ ]:
# ─── 5-B: User steering — refocus the summary ─────────────────────────────────
# Same thread_id: LangGraph loads the checkpoint and continues.
# The summarizer will see the full prior history + this new instruction.

print("=== Turn 2: User requests battery/range focus ===")
r2 = app.invoke(
    {"messages": [HumanMessage("Focus more on the battery specifications and riding range.")]},
    config_mt,
)

final_ai_2 = next(m for m in reversed(r2["messages"]) if isinstance(m, AIMessage))
print("\nRevised summary:")
print(final_ai_2.content)
print(f"\nState now has {len(r2['messages'])} messages accumulated.")

In [ ]:
# ─── 5-C: Second steering turn — remove specific details ──────────────────────

print("=== Turn 3: User removes a detail ===")
r3 = app.invoke(
    {"messages": [HumanMessage("Remove the price from the summary — keep everything else.")]},
    config_mt,
)

final_ai_3 = next(m for m in reversed(r3["messages"]) if isinstance(m, AIMessage))
print("\nFinal summary after steering:")
print(final_ai_3.content)

print("\n=== Price still in summary? ===")
has_price = any(p in final_ai_3.content for p in ["1,299", "EUR", "price", "Price"])
print(f"Price mentioned: {has_price} (expected: False)")

---

## Part 6 — Stopping Conditions and Quality Gates

---

### Stopping Condition Strategies

| Strategy | How | When to use |
|----------|-----|-------------|
| **Round cap** | `len(messages) > N` | Default — prevents infinite loops |
| **Keyword gate** | Check if reviewer output contains "APPROVED" | When the reviewer can signal satisfaction |
| **Score gate** | Parse a numeric score from reviewer output | When quality is measurable |
| **Round counter in state** | Explicit `round: int` field + `round >= max_rounds` | Cleaner than message count arithmetic |

The production code uses **round cap** via message count: `len(messages) > 4` stops after 2 full critic rounds.

In [ ]:
# ─── 6-A: Round cap comparison — 1 round vs 2 rounds vs 3 rounds ──────────────
# Rebuild the graph with configurable round limits to compare output quality.


def make_graph(max_rounds: int):
    """Build a critic loop graph with a configurable round limit."""

    def _should_continue(state: SummaryAgentState) -> bool:
        # 1 HumanMessage + 2 messages/round
        # stop when accumulated messages exceed 2 * max_rounds
        return len(state["messages"]) <= (2 * max_rounds - 1)

    g = StateGraph(SummaryAgentState)
    g.add_node("summarizer", generate_summary)
    g.add_node("reviewer",   review_summary)
    g.add_conditional_edges("summarizer", _should_continue, {True: "reviewer", False: END})
    g.add_edge("reviewer", "summarizer")
    g.set_entry_point("summarizer")
    return g.compile()


print("Comparing 1 round vs 2 rounds vs 3 rounds:\n")
for rounds in [1, 2, 3]:
    g = make_graph(rounds)
    r = g.invoke({"messages": [HumanMessage(SOURCE_TEXT)]})
    final = next(m.content for m in reversed(r["messages"]) if isinstance(m, AIMessage))
    spec_hits = sum(1 for v in spec_values if v in final)
    print(f"Rounds={rounds}: {spec_hits}/{len(spec_values)} spec values | {len(final)} chars")
    print(f"  {final[:120]}...\n")

In [ ]:
# ─── 6-B: Keyword gate — stop when reviewer says APPROVED ─────────────────────
# A more semantic stopping condition: the reviewer explicitly signals satisfaction.

STRICT_REVIEWER_PROMPT = (
    "You are a strict quality reviewer. "
    "Compare the source document and the generated summary. "
    "If the summary includes at least 4 specific measurements or numbers from the source, "
    "respond with exactly: 'APPROVED: <brief reason>' "
    "Otherwise respond with specific improvement suggestions in under 50 words."
)


def review_strict(state: SummaryAgentState) -> dict:
    messages = [SystemMessage(STRICT_REVIEWER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    print(f"  [strict reviewer] {result.content[:120]}")
    return {"messages": [result]}


def should_continue_strict(state: SummaryAgentState) -> bool:
    """Stop if the reviewer approved OR we've hit 3 rounds."""
    if len(state["messages"]) > 6:  # hard cap at 3 rounds
        return False
    # Check if the last reviewer message contains APPROVED
    ai_msgs = [m for m in state["messages"] if isinstance(m, AIMessage)]
    if len(ai_msgs) >= 2:
        last_reviewer = ai_msgs[-1]  # reviewer is always second in each pair
        if "APPROVED" in last_reviewer.content:
            print("  [gate] Reviewer approved — stopping loop.")
            return False
    return True


g_strict = StateGraph(SummaryAgentState)
g_strict.add_node("summarizer", generate_summary)
g_strict.add_node("reviewer",   review_strict)
g_strict.add_conditional_edges("summarizer", should_continue_strict, {True: "reviewer", False: END})
g_strict.add_edge("reviewer", "summarizer")
g_strict.set_entry_point("summarizer")
strict_app = g_strict.compile()

print("Running strict reviewer (stops on APPROVED or 3 rounds):\n")
strict_result = strict_app.invoke({"messages": [HumanMessage(SOURCE_TEXT)]})

final_strict = next(m.content for m in reversed(strict_result["messages"]) if isinstance(m, AIMessage))
print(f"\nFinal ({len(strict_result['messages'])} messages total):")
print(final_strict)

### Exercise 2 — Round Counter in State

The current stopping condition uses message count arithmetic to infer which round we're on. A cleaner approach is to add an explicit `round` counter to the state.

**Task:** Extend `SummaryAgentState` to include a `round: int` field. Increment it in `generate_summary`. Change `should_continue` to check `state["round"] >= max_rounds`.

**Hint:** For a plain `int` field (not a list), the default LangGraph reducer **replaces** the value — which is exactly what you want for a counter. Initialize with `{"messages": [...], "round": 0}` in your invoke call.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
MAX_ROUNDS = 3


class SummaryAgentStateWithCounter(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    round: int  # TODO: increment this in generate_summary


def generate_summary_counted(state: SummaryAgentStateWithCounter) -> dict:
    # TODO: increment state["round"] and print which round we're on
    messages = [SystemMessage(SUMMARIZER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    return {"messages": [result]}  # TODO: also return {"round": state["round"] + 1}


def should_continue_counted(state: SummaryAgentStateWithCounter) -> bool:
    # TODO: return state.get("round", 0) < MAX_ROUNDS
    pass


# TODO: build the graph using SummaryAgentStateWithCounter
# TODO: run it and verify the round counter stops at MAX_ROUNDS

print("Implement the round counter and run the graph.")
print(f"Target: loop should stop after {MAX_ROUNDS} rounds.")

### Exercise 3 — Different Reviewer Persona

The reviewer persona has a major effect on what the summarizer improves. Try one of these:

1. **Technical reviewer**: `"Reject unless the summary includes at least 3 specific technical measurements (voltage, wattage, weight, etc.)."`
2. **Marketing reviewer**: `"Evaluate the summary as a marketing tagline. Reject unless it is persuasive and highlights a key user benefit."`
3. **Brevity reviewer**: `"Reject unless the summary is under 20 words."`

Replace the reviewer prompt in the cell below and observe how the final summary changes.

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────
# TODO: pick one of the reviewer personas above and paste it here
MY_REVIEWER_PROMPT = (
    "TODO: replace with your chosen reviewer persona"
)

# TODO: build a graph with your custom reviewer and run it on SOURCE_TEXT
# Observe how the final summary differs from the default 2-round result

print("Replace MY_REVIEWER_PROMPT and build the graph.")

---

## Part 7 ★ — Advanced Patterns (Bonus)

---

### Pattern 1 — Async Streaming

For production use in a web server you want to stream partial outputs rather than blocking until the full graph completes:

```python
async for event in app.astream_events(
    {"messages": [HumanMessage(SOURCE_TEXT)]},
    config,
    version="v1",
):
    if event["event"] == "on_chat_model_stream":
        chunk = event["data"]["chunk"]
        print(chunk.content, end="", flush=True)
```

### Pattern 2 — LangSmith Observability

Add two environment variables and every invoke is automatically traced:

```bash
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__...
```

The LangSmith dashboard shows: node execution order, token counts per node, latency per round, and full message history.

### Pattern 3 — Persistent Checkpointing (Production)

`MemorySaver` is in-memory only — it disappears when the process restarts. For production, swap in `SqliteSaver` or `PostgresSaver`:

```python
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string("./checkpoints.db") as checkpointer:
    app = graph_builder.compile(checkpointer=checkpointer)
    # All thread states now survive process restarts
```

In [ ]:
# ─── 7-A: Stream node outputs as they arrive ──────────────────────────────────
# Use stream() instead of invoke() to see each node's output as it completes.
# Full token streaming requires astream_events — shown in the pattern above.

stream_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Streaming node-by-node outputs:\n")
for step in app.stream(
    {"messages": [HumanMessage(SOURCE_TEXT)]},
    stream_config,
):
    # step is a dict: {node_name: partial_state}
    for node_name, state_update in step.items():
        if "messages" in state_update and state_update["messages"]:
            msg = state_update["messages"][-1]
            if isinstance(msg, AIMessage):
                print(f"[{node_name}] {msg.content[:100]}...")

print("\nStreaming complete.")

In [ ]:
# ─── 7-B: Inspect the saved checkpoint ───────────────────────────────────────
# After invoke(), MemorySaver holds the full state — you can inspect it directly.

saved_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
app.invoke({"messages": [HumanMessage(SOURCE_TEXT)]}, saved_config)

# Get the checkpoint
checkpoint = app.get_state(saved_config)

print("Checkpoint inspection:")
print(f"  Thread ID:         {saved_config['configurable']['thread_id']}")
print(f"  Messages in state: {len(checkpoint.values['messages'])}")
print(f"  Next node(s):      {checkpoint.next}  (empty = graph has ended)")
print()
print("Message types in checkpoint:")
for i, m in enumerate(checkpoint.values["messages"]):
    print(f"  [{i}] {type(m).__name__}: {m.content[:60]}...")

---

## What's Next?

You now understand the critic loop pattern, accumulating state, conditional edges, and multi-turn steering. Here's where to go deeper:

### Simpler single-agent loops
- **Example 4** `4-basic-react-agent` — a single ReAct agent with tool calling and no critic. The entry point to understanding LangGraph agent nodes before adding a second agent.

### More sophisticated multi-agent patterns
- **Example 6** `6-multi-agent-graph` ([`../6-multi-agent-graph/`](../6-multi-agent-graph/)) — routing between specialist agents (researcher, writer, editor) based on task type rather than a fixed cycle.
- **Example 18** `18-self-reflecting-agent` ([`../18-self-reflecting-agent/`](../18-self-reflecting-agent/)) — reflexion: a single agent critiques its own output without a separate critic node.
- **Example 11** `11-hitl-approval` ([`../11-hitl-approval/`](../11-hitl-approval/)) — human-in-the-loop: pause the graph at a checkpoint and wait for a human approval before continuing.

### Production considerations
- **Async streaming**: use `astream_events` for token-level streaming in FastAPI/Starlette
- **Persistent checkpointing**: swap `MemorySaver` for `SqliteSaver` or `PostgresSaver` for multi-session persistence
- **LangSmith tracing**: two env vars and every graph run is automatically traced with node-level latency and token counts
- **Evaluation**: run the final summaries through a judge LLM to score factual coverage — see Example 16 for the RAGAS evaluation pattern

### Further reading
- Shinn et al. (2023). *Reflexion: Language Agents with Verbal Reinforcement Learning.* https://arxiv.org/abs/2303.11366
- LangGraph persistence and memory guide: https://langchain-ai.github.io/langgraph/concepts/persistence/
- LangGraph `add_conditional_edges` reference: https://langchain-ai.github.io/langgraph/reference/graphs/

---

*Workshop complete. The next step is Example 6 — routing between specialist agents.*

---

## Exercise Answer Key

Use this section after attempting the exercises. These are sample solutions, not the only correct answers.

### Exercise 1 — Change the Source Document

**What to look for:**
- With a short source (2-3 sentences), the critic loop often converges quickly because there are few facts to verify. The reviewer may give vague feedback like "good summary."
- With a dense spec document, the reviewer typically catches missing numbers or vague phrasing that the summarizer then corrects in round 2.
- The spec value coverage metric from cell 4-C is a lightweight proxy for completeness — use it to quantify improvement across drafts.

**Expected pattern:** Draft 1 captures the broad topic. Draft 2 adds specifics prompted by the reviewer. Draft 3 (if 3 rounds) refines phrasing.

In [ ]:
# Exercise 1 — Sample solution with a different source document
sample_source = (
    "Falcon 9 Block 5 Specification\n"
    "SpaceX Falcon 9 Block 5 is a partially reusable two-stage-to-orbit rocket. "
    "Height: 70m. Mass: 549,054 kg. Payload to LEO: 22,800 kg. "
    "First stage: 9 Merlin engines, 7,607 kN thrust at sea level. "
    "Reusability: first stage lands autonomously on drone ships. "
    "Record: 19 flights on a single booster. Cost per launch: ~$67M."
)

ex1_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
ex1_result = app.invoke({"messages": [HumanMessage(sample_source)]}, ex1_config)

ex1_final = next(m.content for m in reversed(ex1_result["messages"]) if isinstance(m, AIMessage))
print("Exercise 1 — final summary for Falcon 9 spec:")
print(ex1_final)

### Exercise 2 — Round Counter in State

**Key insight:** For a `round: int` field (not `Annotated[..., operator.add]`), LangGraph's default reducer **replaces** the value. So returning `{"round": state["round"] + 1}` from `generate_summary` correctly increments the counter each summarizer turn.

The stopping condition becomes cleaner and more readable: `state["round"] >= max_rounds` vs `len(state["messages"]) > 2 * max_rounds - 1`.

**Edge case:** Remember to initialize `round` to `0` in the invoke call: `app.invoke({"messages": [...], "round": 0}, config)`.

In [ ]:
# Exercise 2 — Sample solution with explicit round counter
EX2_MAX_ROUNDS = 2


class SummaryAgentStateV2(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    round: int  # plain int — replaced (not appended) each update


def generate_summary_v2(state: SummaryAgentStateV2) -> dict:
    current_round = state.get("round", 0) + 1
    print(f"  [summarizer round {current_round}]")
    messages = [SystemMessage(SUMMARIZER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    print(f"    {result.content[:80]}...")
    return {"messages": [result], "round": current_round}


def should_continue_v2(state: SummaryAgentStateV2) -> bool:
    return state.get("round", 0) < EX2_MAX_ROUNDS


g2 = StateGraph(SummaryAgentStateV2)
g2.add_node("summarizer", generate_summary_v2)
g2.add_node("reviewer",   review_summary)
g2.add_conditional_edges("summarizer", should_continue_v2, {True: "reviewer", False: END})
g2.add_edge("reviewer", "summarizer")
g2.set_entry_point("summarizer")
app_v2 = g2.compile()

print(f"Running with explicit round counter (max={EX2_MAX_ROUNDS} rounds):\n")
r_v2 = app_v2.invoke({"messages": [HumanMessage(SOURCE_TEXT)], "round": 0})

final_v2 = next(m.content for m in reversed(r_v2["messages"]) if isinstance(m, AIMessage))
print(f"\nRound counter at end: {r_v2['round']}")
print(f"Messages in state: {len(r_v2['messages'])}")
print(f"Final summary: {final_v2}")

### Exercise 3 — Different Reviewer Persona

**What to expect:**
- **Technical reviewer**: Draft 1 is usually vague; Draft 2 includes more numbers; Draft 3 converges on spec-dense output. Works well with the EcoSprint spec.
- **Marketing reviewer**: The summary pivots from informational to persuasive. Specific numbers may be dropped in favor of benefit language ("60-80km of freedom").
- **Brevity reviewer**: The graph often needs more rounds because compressing to 20 words without losing essential facts is hard. The reviewer may reject multiple drafts.

**Key lesson:** The reviewer persona is the highest-leverage tuning parameter in the critic loop. The generator adapts to whatever the critic values.

In [ ]:
# Exercise 3 — Sample solution: brevity reviewer
BREVITY_REVIEWER_PROMPT = (
    "You are a brevity reviewer. The summary must be 20 words or fewer. "
    "Count the exact word count. If it is over 20 words, reject it and say "
    "'TOO LONG (<N> words): remove [specific phrase]'. "
    "If it is 20 words or fewer, say 'APPROVED (<N> words).'."
)


def review_brevity(state: SummaryAgentState) -> dict:
    messages = [SystemMessage(BREVITY_REVIEWER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    print(f"  [brevity reviewer] {result.content}")
    return {"messages": [result]}


g_brevity = StateGraph(SummaryAgentState)
g_brevity.add_node("summarizer", generate_summary)
g_brevity.add_node("reviewer",   review_brevity)
g_brevity.add_conditional_edges("summarizer", should_continue, {True: "reviewer", False: END})
g_brevity.add_edge("reviewer", "summarizer")
g_brevity.set_entry_point("summarizer")
brevity_app = g_brevity.compile()

print("Running brevity reviewer (target: 20 words or fewer):\n")
r_brevity = brevity_app.invoke({"messages": [HumanMessage(SOURCE_TEXT)]})

final_brevity = next(m.content for m in reversed(r_brevity["messages"]) if isinstance(m, AIMessage))
word_count = len(final_brevity.split())
print(f"\nFinal summary ({word_count} words):")
print(final_brevity)